In [1]:
from Age_BMI_loading import age_matrix_vec
age_matrix_vec_2023 = age_matrix_vec


Loaded: ../future_data_1990_2050/age_matrix/age_matrix_0.npy with shape (12, 106729, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_1.npy with shape (12, 110600, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_2.npy with shape (12, 30009, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_3.npy with shape (12, 30156, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_4.npy with shape (12, 18270, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_5.npy with shape (12, 15707, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_6.npy with shape (12, 6451, 61)
Loaded: ../future_data_1990_2050/age_matrix/age_matrix_7.npy with shape (12, 7258, 61)


In [2]:
import numpy as np
import pandas as pd


In [3]:
import os
import numpy as np
import glob

# Function to read and process forecast data from all simulation folders
def load_forecast_data_from_all_simulations(forecast_base_dir = "../../output/Forecast"):
    """
    Load forecast data from all simulation folders under ../../output/Forecast/
    Returns a list of 8 3D arrays, where each array has shape (n_simulations, n_rows, n_cols)
    """
    
    
    # Get all simulation folders (folders with timestamp pattern)
    simulation_folders = glob.glob(os.path.join(forecast_base_dir, "*"))
   
    
    print(f"Found {len(simulation_folders)} simulation folders")
    
    # First pass: determine the maximum number of rows for each matrix (0 to 7)
    max_rows_per_matrix = [0] * 8
    num_cols = 27  # Columns are always the same
    
    for sim_folder in simulation_folders:
        for matrix_idx in range(8):
            filename = os.path.join(sim_folder, f"popu/forecast_matrix_{matrix_idx}.bin")
            
            if os.path.exists(filename):
                # Get file size to determine dimensions
                file_size = os.path.getsize(filename)
                num_rows = file_size // (num_cols * 4)  # 4 bytes per int32
                max_rows_per_matrix[matrix_idx] = max(max_rows_per_matrix[matrix_idx], num_rows)
    
    print(f"Maximum rows for each matrix: {max_rows_per_matrix}")
    print(f"Number of columns (constant): {num_cols}")
    
    # Initialize list to store 3D arrays for each of the 8 matrices
    age_matrix_vec_2050_3d = []
    
    # Process each matrix index (0 to 7)
    for matrix_idx in range(8):
        matrix_list = []  # Store matrices from all simulations for this index
        max_rows = max_rows_per_matrix[matrix_idx]  # Get max rows for this specific matrix
        
        # Process each simulation folder
        for sim_folder in simulation_folders:
            filename = os.path.join(sim_folder, f"popu/forecast_matrix_{matrix_idx}.bin")
            
            if os.path.exists(filename):
                # Get file size to determine dimensions
                file_size = os.path.getsize(filename)
                sim_rows = file_size // (num_cols * 4)  # 4 bytes per int32
                
                # Read binary data
                with open(filename, 'rb') as f:
                    data = np.fromfile(f, dtype=np.int32)
                    matrix = data.reshape((sim_rows, num_cols))
                
                # Pad matrix to max_rows if needed
                # Create padding filled with -1
                padding_rows = max_rows - sim_rows
                padding = np.full((padding_rows, num_cols), -1, dtype=np.int32)
                matrix = np.vstack([matrix, padding])
                
                matrix_list.append(matrix)
            else:
                print(f"Warning: File {filename} not found")
        
        # Convert list to 3D numpy array
        if matrix_list:
            # Stack all matrices from different simulations
            matrix_3d = np.stack(matrix_list, axis=0)  # Shape: (n_simulations, n_rows, n_cols)
            age_matrix_vec_2050_3d.append(matrix_3d)
            print(f"Matrix {matrix_idx}: Shape {matrix_3d.shape} (max_rows={max_rows})")
        else:
            print(f"Warning: No data found for matrix {matrix_idx}")
            age_matrix_vec_2050_3d.append(np.array([]))
    
    return age_matrix_vec_2050_3d

# Load the 3D forecast data
age_matrix_vec_2050 = load_forecast_data_from_all_simulations()


Found 10 simulation folders
Maximum rows for each matrix: [90488, 96945, 19031, 18890, 13064, 12125, 5314, 6508]
Number of columns (constant): 27
Matrix 0: Shape (10, 90488, 27) (max_rows=90488)
Matrix 1: Shape (10, 96945, 27) (max_rows=96945)
Matrix 2: Shape (10, 19031, 27) (max_rows=19031)
Matrix 3: Shape (10, 18890, 27) (max_rows=18890)
Matrix 4: Shape (10, 13064, 27) (max_rows=13064)
Matrix 5: Shape (10, 12125, 27) (max_rows=12125)
Matrix 6: Shape (10, 5314, 27) (max_rows=5314)
Matrix 7: Shape (10, 6508, 27) (max_rows=6508)


In [5]:
# Check the shape and analyze zeros in the 4th matrix (index 3) from the first simulation
matrix_to_analyze = age_matrix_vec_2050[3][0]
print(f"Analyzing matrix shape: {matrix_to_analyze.shape}")
print(f"Matrix type: {type(matrix_to_analyze)}")

# Count zeros in each column
zero_counts = np.count_nonzero(matrix_to_analyze == 0, axis=0)

zero_counts

# Display summary
# print(f"\nSummary of zero counts across all {num_cols} columns:")
# print(f"Total zeros per column: {zero_counts}")
# print(f"Min zeros in any column: {min(zero_counts)}")
# print(f"Max zeros in any column: {max(zero_counts)}")
# print(f"Average zeros per column: {np.mean(zero_counts):.2f}")

# Also check for -1 values (which seem to be used as padding/missing values)
# print(f"\nAlso checking for -1 values (padding):")
# neg_one_counts = []
# for col_idx in range(num_cols):
#     neg_one_count = np.sum(matrix_to_analyze[:, col_idx] == -1)
#     neg_one_counts.append(neg_one_count)
#     print(f"Column {col_idx}: {neg_one_count} values of -1")


Analyzing matrix shape: (18890, 27)
Matrix type: <class 'numpy.ndarray'>


array([201, 178, 195, 193, 182, 189, 186, 170, 172, 156, 165, 156, 156,
       133, 143, 129, 139, 133, 113, 125, 126, 120, 119, 105, 113, 128,
       126])

In [ ]:
# def continue_with_alive(historical_matrix, forecast_matrix):
#     """
#     Perform left join between two matrices based on last column of first matrix 
#     and first column of second matrix (with +1 offset).
    
#     Args:
#         historical_matrix: First matrix (2D numpy array)
#         forecast_matrix: Second matrix (2D numpy array)
        
#     Returns:
#         left_joined_matrix: Concatenated matrix from left join operation
#     """
#     rows_last_col_not_zero = historical_matrix[historical_matrix[:, -1] != -1]
    
#     # when doing the forecast, the first rows_last_col_not_zero.shape[0] rows are exasctly from the previous one
#     forecast_matrix = forecast_matrix[:rows_last_col_not_zero.shape[0], ]

#     # Find range of values in last column of rows_last_col_not_zero
#     unique_values = np.unique(rows_last_col_not_zero[:, -1])
#     print(f"Range of values in last column: {unique_values.min()} to {unique_values.max()}")
#     print(f"Unique values: {unique_values}")

#     # Perform left join logic
#     result_rows = []
    
#     # Case 1: Equal sizes
#     for value in unique_values:
#         # Find rows in first matrix where last column equals value
#         first_matrix_rows = rows_last_col_not_zero[rows_last_col_not_zero[:, -1] == value]
        
#         # Find rows in second matrix where first column equals value + 1
#         second_matrix_rows = forecast_matrix[forecast_matrix[:, 0] == (value + 1)]
        
#         if len(first_matrix_rows) == len(second_matrix_rows):
#             # Block concatenation when sizes match
#             joined_block = np.concatenate([first_matrix_rows, second_matrix_rows], axis=1)
#             result_rows.extend(joined_block)
    
#     # Count zeros in each column of the result matrix so far
#     if result_rows:
#         mat = np.array(result_rows)
#         zero_counts = np.sum(mat == 0, axis=0)
#         print(f"Zero counts per column in result matrix: {zero_counts}")
    
#     # Case 2: First matrix has fewer rows than second matrix
#     for value in unique_values:
#         # Find rows in first matrix where last column equals value
#         first_matrix_rows = rows_last_col_not_zero[rows_last_col_not_zero[:, -1] == value]
        
#         # Find rows in second matrix where first column equals value + 1
#         second_matrix_rows = forecast_matrix[forecast_matrix[:, 0] == (value + 1)]
        
#         if len(first_matrix_rows) < len(second_matrix_rows):
#             # Handle case where first matrix has fewer rows than second matrix
#             # This shouldn't happen in normal cases, but we handle it 
#             if len(first_matrix_rows) > 0:
#                 matched_second = second_matrix_rows[:len(first_matrix_rows)]
#                 joined_block = np.concatenate([first_matrix_rows, matched_second], axis=1)
#                 result_rows.extend(joined_block)
    
#     if result_rows:
#         mat = np.array(result_rows)
#         zero_counts = np.sum(mat == 0, axis=0)
#         print(f"Zero counts per column in result matrix: {zero_counts}")
    
#     # Case 3: First matrix has more rows than second matrix  
#     for value in unique_values:
#         # Find rows in first matrix where last column equals value
#         first_matrix_rows = rows_last_col_not_zero[rows_last_col_not_zero[:, -1] == value]
        
#         # Find rows in second matrix where first column equals value + 1
#         second_matrix_rows = forecast_matrix[forecast_matrix[:, 0] == (value + 1)]
        
#         if len(first_matrix_rows) > len(second_matrix_rows):
#             # len(first_matrix_rows) > len(second_matrix_rows)
#             # Block join for available matches
#             if len(second_matrix_rows) > 0:
#                 matched_first = first_matrix_rows[:len(second_matrix_rows)]
#                 joined_block = np.concatenate([matched_first, second_matrix_rows], axis=1)
#                 result_rows.extend(joined_block)
#                 unmatched_first = first_matrix_rows[len(second_matrix_rows):]
#                 padding = np.full((len(unmatched_first), forecast_matrix.shape[1]), -1)
#                 unmatched_block = np.concatenate([unmatched_first, padding], axis=1)
#                 result_rows.extend(unmatched_block)
#     if result_rows:
#         mat = np.array(result_rows)
#         zero_counts = np.sum(mat == 0, axis=0)
#         print(f"Zero counts per column in result matrix: {zero_counts}")

#     # Convert to numpy array
#     left_joined_matrix = np.array(result_rows)
#     return left_joined_matrix

# # step 1 



In [6]:
def continue_with_alive(historical_matrix, forecast_matrix):
    """
    Perform left join between two matrices based on last column of first matrix 
    and first column of second matrix (with +1 offset).
    
    Args:
        historical_matrix: First matrix (2D numpy array)
        forecast_matrix: Second matrix (2D numpy array)
        
    Returns:
        left_joined_matrix: Concatenated matrix from left join operation
    """
    rows_last_col_not_zero = historical_matrix[historical_matrix[:, -1] != -1]
    
    # when doing the forecast, the first rows_last_col_not_zero.shape[0] rows are exasctly from the previous one
    forecast_matrix = forecast_matrix[:rows_last_col_not_zero.shape[0], ]

    # Find range of values in last column of rows_last_col_not_zero
    unique_values = np.unique(rows_last_col_not_zero[:, -1])
    print(f"Range of values in last column: {unique_values.min()} to {unique_values.max()}")
    print(f"Unique values: {unique_values}")

    # Perform left join logic
    result_rows = []
    for value in unique_values:
        # Find rows in first matrix where last column equals value
        first_matrix_rows = rows_last_col_not_zero[rows_last_col_not_zero[:, -1] == value]
        
        # Find rows in second matrix where first column equals value + 1
        second_matrix_rows = forecast_matrix[forecast_matrix[:, 0] == (value + 1)]
        
        #print(f"Value {value}: First matrix has {len(first_matrix_rows)} rows, Second matrix has {len(second_matrix_rows)} rows")
        
        # Check if first matrix has fewer rows than second matrix (should not happen)
        # if len(first_matrix_rows) < len(second_matrix_rows):
        #     print(f"WARNING: First matrix has fewer rows ({len(first_matrix_rows)}) than second matrix ({len(second_matrix_rows)}) for value {value}")
        
        # Efficient left join based on matrix sizes
        if len(first_matrix_rows) == len(second_matrix_rows):
            # Block concatenation when sizes match
            joined_block = np.concatenate([first_matrix_rows, second_matrix_rows], axis=1)
            result_rows.extend(joined_block)
        elif len(first_matrix_rows) < len(second_matrix_rows):
            # Handle case where first matrix has fewer rows than second matrix
            # This shouldn't happen in normal cases, but we handle it 
            if len(first_matrix_rows) >= 0:
                matched_second = second_matrix_rows[:len(first_matrix_rows)]
                joined_block = np.concatenate([first_matrix_rows, matched_second], axis=1)
                result_rows.extend(joined_block)
        elif len(first_matrix_rows) > len(second_matrix_rows):
            # len(first_matrix_rows) > len(second_matrix_rows)
            # Block join for available matches
            if len(second_matrix_rows) >= 0:
                matched_first = first_matrix_rows[:len(second_matrix_rows)]
                joined_block = np.concatenate([matched_first, second_matrix_rows], axis=1)
                result_rows.extend(joined_block)
                unmatched_first = first_matrix_rows[len(second_matrix_rows):]
                padding = np.full((len(unmatched_first), forecast_matrix.shape[1]), -1)
                unmatched_block = np.concatenate([unmatched_first, padding], axis=1)
                result_rows.extend(unmatched_block)

    # Convert to numpy array
    left_joined_matrix = np.array(result_rows)
    return left_joined_matrix

# step 1 



In [17]:
# left_joined_matrix = continue_with_alive(age_matrix_vec_2023[2], age_matrix_vec_2050[2])

In [7]:
def continue_with_death(historical_matrix, forecast_matrix):
    """
    Create a right joined matrix by taking the remaining part of the forecast matrix
    after histo_matrix.shape[0] rows and padding with -1 values on the left.
    
    Args:
        histo_matrix: The historical matrix used to determine the cutoff point
        forecast_matrix: The full forecast matrix
        
    Returns:
        right_joined_matrix: The resulting right joined matrix
    """
    rows_last_col_not_zero = historical_matrix[historical_matrix[:, -1] != -1]
    #rows_last_col_not_zero = historical_matrix
    # Get the remaining part of the age matrix after histo_matrix.shape[0] rows
    remaining_matrix = forecast_matrix[rows_last_col_not_zero.shape[0]:, ]

    # Create an array with same row length as remaining_matrix, 34 columns, filled with -1
    padding_matrix = np.full((remaining_matrix.shape[0], 34), -1)

    # Right join: concatenate padding_matrix (left) with remaining_matrix (right)
    right_joined_matrix = np.concatenate([padding_matrix, remaining_matrix], axis=1)

    print(f"Remaining matrix shape: {remaining_matrix.shape}")
    print(f"Padding matrix shape: {padding_matrix.shape}")
    print(f"Right joined matrix shape: {right_joined_matrix.shape}")
    
    return right_joined_matrix

# second part;
# right_joined_matrix = continue_with_death(second_matrix, age_matrix_vec_2050[0])

In [8]:
def add_new_people(matrix_0, padding_columns=27):
    """
    Create a left joined matrix for new people by filtering rows where the last column is -1
    and padding with -1 values on the right.
    
    Args:
        matrix_0: The input matrix to filter
        padding_columns: Number of columns to pad with -1 values (default: 27)
        
    Returns:
        left_joined_matrix_27: The resulting left joined matrix with padding for new people
    """
    # Filter rows where the last column is -1
    rows_last_col_zero = matrix_0[matrix_0[:, -1] == -1]

    # Create a matrix with specified columns filled with -1, same row size as filtered rows
    padding_matrix_27 = np.full((rows_last_col_zero.shape[0], padding_columns), -1)

    # Left join: concatenate rows_last_col_zero (left) with padding_matrix_27 (right)
    left_joined_matrix_27 = np.concatenate([rows_last_col_zero, padding_matrix_27], axis=1)

    print(f"rows_last_col_zero shape: {rows_last_col_zero.shape}")
    print(f"Padding matrix ({padding_columns} cols) shape: {padding_matrix_27.shape}")
    print(f"Left joined matrix ({padding_columns} cols) shape: {left_joined_matrix_27.shape}")
    
    return left_joined_matrix_27

# third part 



In [22]:
# left_joined_matrix_27 = add_new_people(matrix_0)

In [24]:
# Vertically stack the three matrices
# final_matrix = np.vstack([left_joined_matrix, right_joined_matrix, left_joined_matrix_27])

# print(f"Final stacked matrix shape: {final_matrix.shape}")
# print(f"Left joined matrix shape: {left_joined_matrix.shape}")
# print(f"Right joined matrix shape: {right_joined_matrix.shape}")
# print(f"Left joined matrix (27 cols) shape: {left_joined_matrix_27.shape}")


In [25]:
age_matrix_vec_2023[0].shape

(88417, 34)

In [9]:

def validate_joined_matrix_age_continuity(left_joined_matrix):
    """
    Validate that the age continuity between the 33rd and 34th columns is correct.
    
    Args:
        left_joined_matrix: Input matrix to validate
        
    Returns:
        bool: True if validation passes, False otherwise
    """
    # Get the 33rd and 34th columns (0-indexed: 32 and 33)
    col_33 = left_joined_matrix[:, 33]  # Last column from first matrix
    col_34 = left_joined_matrix[:, 34]  # First column from second matrix

    # Find rows where 34th column is not -1
    valid_rows_mask = col_34 != -1
    valid_rows_indices = np.where(valid_rows_mask)[0]

    print(f"Number of rows where 34th column is not -1: {len(valid_rows_indices)}")

    if len(valid_rows_indices) > 0:
        # Check if 33rd column is always 1 smaller than 34th column when 34th is not -1
        col_33_valid = col_33[valid_rows_mask]
        col_34_valid = col_34[valid_rows_mask]
        
        expected_relationship = col_33_valid == (col_34_valid - 1)
        
        print(f"Number of rows satisfying the relationship (33rd = 34th - 1): {np.sum(expected_relationship)}")
        print(f"Total valid rows: {len(col_33_valid)}")
        print(f"Validation passed: {np.all(expected_relationship)}")
        
        # Show some examples
        print("\nFirst 10 examples:")
        for i in range(min(10, len(valid_rows_indices))):
            idx = valid_rows_indices[i]
            print(f"Row {idx}: 33rd col = {col_33[idx]}, 34th col = {col_34[idx]}, Difference = {col_34[idx] - col_33[idx]}")
        
        # If validation fails, show problematic rows
        if not np.all(expected_relationship):
            problem_indices = valid_rows_indices[~expected_relationship]
            print(f"\nProblematic rows (first 5): {problem_indices[:5]}")
            for idx in problem_indices[:5]:
                print(f"Row {idx}: 33rd col = {col_33[idx]}, 34th col = {col_34[idx]}, Difference = {col_34[idx] - col_33[idx]}")
            return False
        return True
    else:
        print("No valid rows found for validation")
        return False

# Call the validation function
# validation_result = validate_joined_matrix_age_continuity(left_joined_matrix)



In [28]:
for i in range(8):

    # Get mat
    mat_2023 = age_matrix_vec_2023[i]
    mat_2050 = age_matrix_vec_2050[i]
    # Print shapes for each matrix
    print(f"Index {i}:")
    print(f"  mat_2023 shape: {mat_2023.shape}")
    print(f"  mat_2050 shape: {mat_2050.shape}")
    print()

Index 0:
  mat_2023 shape: (88417, 34)
  mat_2050 shape: (12, 93313, 27)

Index 1:
  mat_2023 shape: (91217, 34)
  mat_2050 shape: (12, 98580, 27)

Index 2:
  mat_2023 shape: (16696, 34)
  mat_2050 shape: (12, 27967, 27)

Index 3:
  mat_2023 shape: (16310, 34)
  mat_2050 shape: (12, 28561, 27)

Index 4:
  mat_2023 shape: (12057, 34)
  mat_2050 shape: (12, 16359, 27)

Index 5:
  mat_2023 shape: (10812, 34)
  mat_2050 shape: (12, 14479, 27)

Index 6:
  mat_2023 shape: (3903, 34)
  mat_2050 shape: (12, 5976, 27)

Index 7:
  mat_2023 shape: (4679, 34)
  mat_2050 shape: (12, 6742, 27)



In [105]:
age_matrix_vec_2023[0].shape

(88417, 34)

In [113]:
age_matrix_vec_2050[0].shape

(12, 93313, 27)

In [109]:
age_matrix_vec[0].shape

(1, 93313, 61)

In [10]:
# Create combined matrices for each index
age_matrix_vec = []
for i in [0]:
    
    # Get mat
    mat_2023 = age_matrix_vec_2023[i]
    mat_2050_3d = age_matrix_vec_2050[i]  # Shape: (n_simulations, n_rows, n_cols)
    
    # Initialize list to store combined matrices for all simulations
    combined_matrices_for_index = []
    
    # Process each simulation
    for sim_idx in range(1):
        mat_2050 = mat_2050_3d[sim_idx]  # Get matrix for this simulation
        print(f"mat_2050 shape: {mat_2050.shape}")
        print(f"mat_2050 zeros per column: {np.sum(mat_2050 == 0, axis=0)}")
        
        print(f"mat_2023 shape: {mat_2023.shape}")
        print(f"mat_2023 zeros per column: {np.sum(mat_2023 == 0, axis=0)}")
        # Apply the three strategies for combining data
        
        # step 1 
        alive_mat = continue_with_alive(mat_2023, mat_2050)
        print(f"alive_mat shape: {alive_mat.shape}")
        print(f"alive_mat zeros per column: {np.sum(alive_mat == 0, axis=0)}")

        # check alive_mat
        validate_joined_matrix_age_continuity(alive_mat)

        # second part;
        death_mat = continue_with_death(mat_2023, mat_2050)
        print(f"death_mat shape: {death_mat.shape}")
        print(f"death_mat zeros per column: {np.sum(death_mat == 0, axis=0)}")

        # third part 
        new_people_mat = add_new_people(mat_2023)
        print(f"new_people_mat shape: {new_people_mat.shape}")
        print(f"new_people_mat zeros per column: {np.sum(new_people_mat == 0, axis=0)}")

        # Combine the three matrices , new_people_mat
        combined = np.vstack([alive_mat, death_mat, new_people_mat])
        
        combined_matrices_for_index.append(combined)
    
    # Stack all simulation results for this matrix index
    combined_3d = np.stack(combined_matrices_for_index, axis=0)  # Shape: (n_simulations, n_rows, n_cols)
    age_matrix_vec.append(combined_3d)

age_matrix_vec

mat_2050 shape: (90488, 27)
mat_2050 zeros per column: [613 528 521 509 470 461 465 414 427 412 357 382 432 366 347 344 338 319
 286 302 287 281 269 278 304 279 277]
mat_2023 shape: (88417, 34)
mat_2023 zeros per column: [834 919 855 885 882 875 854 833 810 734 741 763 657 648 599 610 613 620
 651 613 584 563 616 632 579 617 606 574 549 553 557 525 523 494]
Range of values in last column: 0 to 118
Unique values: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 106 107 108
 110 111 112 114 118]
alive_mat shape: (74999, 61)
alive_mat zeros per column: [817 919 844 881 880 875 850 816 793 690 732 703 609 633 577 592 58

[array([[[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]]])]

In [120]:
age_matrix_vec[0].shape

(1, 106731, 61)

In [95]:
temp = np.sum(combined == 0, axis=0)

temp

array([817, 919, 844, 881, 880, 875, 850, 816, 793, 690, 732, 703, 609,
       633, 577, 592, 587, 600, 635, 586, 577, 557, 608, 611, 565, 609,
       605, 574, 548, 551, 556, 524, 523, 494, 660, 535, 542, 484, 484,
       477, 453, 450, 436, 417, 360, 365, 413, 365, 382, 339, 327, 325,
       292, 324, 302, 299, 264, 282, 313, 269, 282])

In [ ]:
247, 259, 240, 251, 252, 233, 225, 221, 210, 211, 192, 201, 187,
       177, 164, 162, 163, 166, 158, 156, 150, 150, 150, 156, 164, 170,
       182, 188, 199, 194, 205, 198, 212, 199, 203, 212, 221, 212, 217,
       209, 206, 227, 212, 220, 211, 204, 217, 228, 230, 228, 237, 221,
       238, 238, 236, 244, 266, 247, 257, 255, 269

In [ ]:
# 排除immigration 干扰试试看

In [51]:
temp = np.sum(age_matrix_vec[0][3] == 0, axis=0)

# Find the index of the first non-zero value in temp
first_non_zero_idx = np.argmax(temp != 0)
print(f"Index of first non-zero value: {first_non_zero_idx}")
print(f"First non-zero value: {temp[first_non_zero_idx]}")


Index of first non-zero value: 0
First non-zero value: 834


In [52]:
temp

array([834, 919, 855, 885, 882, 875, 854, 833, 810, 734, 741, 763, 657,
       648, 599, 610, 613, 620, 651, 613, 584, 563, 616, 632, 579, 617,
       606, 574, 549, 553, 557, 525, 523, 494,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0, 118, 413, 365, 382, 339, 327, 325,
       292, 324, 302, 299, 264, 282, 313, 269, 282])

In [11]:
# Create combined matrices for each index
age_matrix_vec = []
for i in range(8):
    
    # Get mat
    mat_2023 = age_matrix_vec_2023[i]
    mat_2050_3d = age_matrix_vec_2050[i]  # Shape: (n_simulations, n_rows, n_cols)
    
    # Initialize list to store combined matrices for all simulations
    combined_matrices_for_index = []
    
    # Process each simulation
    for sim_idx in range(mat_2050_3d.shape[0]):
        mat_2050 = mat_2050_3d[sim_idx]  # Get matrix for this simulation
        
        # Apply the three strategies for combining data
        
        # step 1 
        alive_mat = continue_with_alive(mat_2023, mat_2050)
        print(f"alive_mat shape: {alive_mat.shape}")
        # Check number of 77s in first column
        count_77_alive = np.sum(alive_mat[:, 0] == 77)
        print(f"alive_mat - count of 77 in first column: {count_77_alive}")

        # check alive_mat
        validate_joined_matrix_age_continuity(alive_mat)

        # second part;
        death_mat = continue_with_death(mat_2023, mat_2050)
        print(f"death_mat shape: {death_mat.shape}")
        # Check number of 77s in first column
        count_77_death = np.sum(death_mat[:, 0] == 77)
        print(f"death_mat - count of 77 in first column: {count_77_death}")

        # third part 
        new_people_mat = add_new_people(mat_2023)
        print(f"new_people_mat shape: {new_people_mat.shape}")
        # Check number of 77s in first column
        count_77_new = np.sum(new_people_mat[:, 0] == 77)
        print(f"new_people_mat - count of 77 in first column: {count_77_new}")

        # Combine the three matrices
        combined = np.vstack([alive_mat, death_mat, new_people_mat])
        print(f"combined shape: {combined.shape}")
        
        combined_matrices_for_index.append(combined)
    
    # Stack all simulation results for this matrix index
    combined_3d = np.stack(combined_matrices_for_index, axis=0)  # Shape: (n_simulations, n_rows, n_cols)
    age_matrix_vec.append(combined_3d)

age_matrix_vec

Range of values in last column: 0 to 118
Unique values: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 106 107 108
 110 111 112 114 118]
alive_mat shape: (74999, 61)
alive_mat - count of 77 in first column: 2
Number of rows where 34th column is not -1: 74437
Number of rows satisfying the relationship (33rd = 34th - 1): 74437
Total valid rows: 74437
Validation passed: True

First 10 examples:
Row 0: 33rd col = 0, 34th col = 1, Difference = 1
Row 1: 33rd col = 0, 34th col = 1, Difference = 1
Row 2: 33rd col = 0, 34th col = 1, Difference = 1
Row 3: 33rd col = 0, 34th col = 1, Difference = 1
Row 4: 33rd col = 0, 34th 

[array([[[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]],
 
        [[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]],
 
        [[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]],
 
        ...,
 
        [[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],


In [173]:
sum(mat_2023[:, 0] == 77)

14

In [171]:
matrix_0 = mat_2023

In [158]:
rows_last_col_zero = matrix_0[matrix_0[:, -1] != -1]

In [ ]:

print(f"Count of rows where first column equals 77: {sum(rows_last_col_zero[:, 0] == 77)}")

1

In [167]:
print(rows_last_col_zero[rows_last_col_zero[:, 0] == 77])

[[ 77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94
   95  96  97  98  99 100 101 102 103 104 105 106 107 108 109 110]]


In [174]:
forecast_matrix =age_matrix_vec_2050[3][0]

In [175]:
sum(forecast_matrix[:, 0] == 77)

91

In [12]:
# Create combined matrices for each index
age_matrix_vec = []
for i in range(8):
    
    # Get mat
    mat_2023 = age_matrix_vec_2023[i]
    mat_2050_3d = age_matrix_vec_2050[i]  # Shape: (n_simulations, n_rows, n_cols)
    
    # Initialize list to store combined matrices for all simulations
    combined_matrices_for_index = []
    
    # Process each simulation
    for sim_idx in range(mat_2050_3d.shape[0]):
        mat_2050 = mat_2050_3d[sim_idx]  # Get matrix for this simulation
        
        # Apply the three strategies for combining data
        
        # step 1 
        alive_mat = continue_with_alive(mat_2023, mat_2050)
        print(f"alive_mat shape: {alive_mat.shape}")

        # check alive_mat
        validate_joined_matrix_age_continuity(alive_mat)

        # second part;
        death_mat = continue_with_death(mat_2023, mat_2050)
        print(f"death_mat shape: {death_mat.shape}")

        # third part 
        new_people_mat = add_new_people(mat_2023)
        print(f"new_people_mat shape: {new_people_mat.shape}")

        # Combine the three matrices
        combined = np.vstack([alive_mat, death_mat, new_people_mat])
        print(f"combined shape: {combined.shape}")
        
        combined_matrices_for_index.append(combined)
    
    # Stack all simulation results for this matrix index
    combined_3d = np.stack(combined_matrices_for_index, axis=0)  # Shape: (n_simulations, n_rows, n_cols)
    age_matrix_vec.append(combined_3d)

age_matrix_vec

Range of values in last column: 0 to 118
Unique values: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 106 107 108
 110 111 112 114 118]
alive_mat shape: (74999, 61)
Number of rows where 34th column is not -1: 74437
Number of rows satisfying the relationship (33rd = 34th - 1): 74437
Total valid rows: 74437
Validation passed: True

First 10 examples:
Row 0: 33rd col = 0, 34th col = 1, Difference = 1
Row 1: 33rd col = 0, 34th col = 1, Difference = 1
Row 2: 33rd col = 0, 34th col = 1, Difference = 1
Row 3: 33rd col = 0, 34th col = 1, Difference = 1
Row 4: 33rd col = 0, 34th col = 1, Difference = 1
Row 5: 33rd col = 0

[array([[[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]],
 
        [[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]],
 
        [[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1]],
 
        ...,
 
        [[-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         [-1, -1, -1, ..., 25, 26, 27],
         ...,
         [-1, -1, -1, ..., -1, -1, -1],
         [-1, -1, -1, ..., -1, -1, -1],


In [170]:
age_matrix_vec[0].shape



(12, 30158, 61)

In [13]:
# Print the shape of each matrix in age_matrix_vec
for i, matrix in enumerate(age_matrix_vec):
    print(f"age_matrix_vec[{i}] shape: {matrix.shape}")

age_matrix_vec[0] shape: (10, 103906, 61)
age_matrix_vec[1] shape: (10, 108966, 61)
age_matrix_vec[2] shape: (10, 21074, 61)
age_matrix_vec[3] shape: (10, 20487, 61)
age_matrix_vec[4] shape: (10, 14978, 61)
age_matrix_vec[5] shape: (10, 13353, 61)
age_matrix_vec[6] shape: (10, 5789, 61)
age_matrix_vec[7] shape: (10, 7024, 61)


In [123]:
age_matrix_vec_2023[0].shape

(88417, 34)

In [124]:
age_matrix_vec[0][0].shape

(106731, 61)

In [131]:
len(age_matrix_vec)

1

In [147]:
age_matrix_vec[3][0]

IndexError: list index out of range

In [177]:
# Find number of values in range 1 to 85 in first column of two matrices
count_2023 = []
count_2050 = []

# Count for age_matrix_vec_2023[0] first column
first_col_2023 = age_matrix_vec_2023[3][:, 0]
for age in range(1, 86):  # 1 to 85 inclusive
    count = np.sum(first_col_2023 == age)
    count_2023.append(count)

# Count for age_matrix_vec[0][0] first column  
first_col_2050 = age_matrix_vec[0][1][:, 0]
for age in range(1, 86):  # 1 to 85 inclusive
    count = np.sum(first_col_2050 == age)
    count_2050.append(count)

print("Count of ages 1-85 in age_matrix_vec_2023[0] first column:")
print(count_2023)
print("\nCount of ages 1-85 in age_matrix_vec[0][0] first column:")
print(count_2050)


Count of ages 1-85 in age_matrix_vec_2023[0] first column:
[231, 225, 205, 201, 199, 185, 186, 172, 163, 143, 138, 130, 125, 128, 139, 145, 161, 161, 161, 148, 173, 184, 213, 225, 225, 233, 226, 226, 223, 216, 205, 197, 183, 178, 170, 162, 147, 139, 123, 123, 112, 113, 115, 66, 57, 72, 60, 60, 68, 73, 76, 63, 51, 64, 83, 75, 64, 58, 51, 57, 54, 59, 54, 37, 35, 31, 41, 40, 21, 22, 21, 25, 25, 9, 10, 9, 14, 15, 6, 6, 4, 5, 6, 1, 12]

Count of ages 1-85 in age_matrix_vec[0][0] first column:
[231, 225, 205, 201, 199, 185, 186, 172, 163, 143, 138, 130, 125, 128, 139, 145, 161, 161, 161, 148, 173, 184, 213, 225, 225, 233, 226, 226, 223, 216, 205, 197, 183, 178, 170, 162, 147, 139, 123, 123, 112, 113, 115, 66, 57, 72, 60, 60, 68, 73, 76, 63, 51, 64, 83, 75, 64, 58, 51, 57, 54, 59, 54, 37, 35, 31, 41, 40, 21, 22, 21, 25, 25, 9, 10, 9, 14, 15, 6, 6, 4, 5, 6, 1, 12]


In [144]:
# Compare count_2023 and count_2050 vectors element by element
differences = []
for i in range(len(count_2023)):
    if count_2023[i] != count_2050[i]:
        differences.append((i+1, count_2023[i], count_2050[i], count_2050[i] - count_2023[i]))

print(f"Found {len(differences)} differences between count_2023 and count_2050:")
print("Age | 2023 | 2050 | Diff")
print("-" * 25)

for age, val_2023, val_2050, diff in differences:
    print(f"{age:3d} | {val_2023:4d} | {val_2050:4d} | {diff:+4d}")

if not differences:
    print("The two vectors are identical")



Found 1 differences between count_2023 and count_2050:
Age | 2023 | 2050 | Diff
-------------------------
 77 |   14 |   13 |   -1


In [14]:
# Save each matrix in age_matrix_vec to individual .npy files
import os

#Create the output directory if it doesn't exist
#ckd/future_data_1990_2050
output_dir = "../future_data_1990_2050/age_matrix"
os.makedirs(output_dir, exist_ok=True)

# Save each matrix as a .npy file
for i in range(len(age_matrix_vec)):
    filename = f"{output_dir}/age_matrix_{i}.npy"
    np.save(filename, age_matrix_vec[i])
    print(f"Saved {filename}")



Saved ../future_data_1990_2050/age_matrix/age_matrix_0.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_1.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_2.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_3.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_4.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_5.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_6.npy
Saved ../future_data_1990_2050/age_matrix/age_matrix_7.npy


In [ ]:

# # Create frequency count for age_matrix_vec_2050[0][:,0] and age_matrix_vec_2023[0][:,-1]
# import matplotlib.pyplot as plt
# from collections import defaultdict

# # Get the two arrays
# arr_2050_start = age_matrix_vec_2050[0][:, 0]  # First column of 2050 data
# arr_2023_end = age_matrix_vec_2023[0][:, -1]   # Last column of 2023 data

# # Create frequency dictionaries for values from -1 to 85
# freq_2050 = defaultdict(int)
# freq_2023 = defaultdict(int)

# # Count frequencies for 2050 data
# for value in arr_2050_start:
#     if -1 <= value <= 85:
#         freq_2050[int(value)] += 1

# # Count frequencies for 2023 data  
# for value in arr_2023_end:
#     if -1 <= value <= 85:
#         freq_2023[int(value)] += 1

# # Convert to regular dictionaries for easier handling
# freq_2050 = dict(freq_2050)
# freq_2023 = dict(freq_2023)

# print("Frequency count for age_matrix_vec_2050[0][:,0]:")
# print(f"Total values: {len(arr_2050_start)}")
# print("Sample frequencies:", {k: v for k, v in list(freq_2050.items())[:10]})

# print("\nFrequency count for age_matrix_vec_2023[0][:,-1]:")
# print(f"Total values: {len(arr_2023_end)}")
# print("Sample frequencies:", {k: v for k, v in list(freq_2023.items())[:10]})

# # Create visualization
# fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))

# # Get all unique values that appear in either dataset
# all_values = sorted(set(list(freq_2050.keys()) + list(freq_2023.keys())))

# # Prepare data for plotting
# values_2050 = [freq_2050.get(val, 0) for val in all_values]
# values_2023 = [freq_2023.get(val, 0) for val in all_values]

# # Plot 1: 2050 data
# ax1.bar(all_values, values_2050, alpha=0.7, color='blue', label='2050 Start')
# ax1.set_title('Frequency Distribution - age_matrix_vec_2050[0][:,0]')
# ax1.set_xlabel('Age Values')
# ax1.set_ylabel('Frequency')
# ax1.legend()
# ax1.grid(True, alpha=0.3)

# # Plot 2: 2023 data
# ax2.bar(all_values, values_2023, alpha=0.7, color='red', label='2023 End')
# ax2.set_title('Frequency Distribution - age_matrix_vec_2023[0][:,-1]')
# ax2.set_xlabel('Age Values')
# ax2.set_ylabel('Frequency')
# ax2.legend()
# ax2.grid(True, alpha=0.3)

# # Plot 3: Difference (2050 - 2023)
# # Filter out values where x = -1
# filtered_values = [val for val in all_values if val != -1]
# filtered_values_2050 = [freq_2050.get(val, 0) for val in filtered_values]
# filtered_values_2023 = [freq_2023.get(val, 0) for val in filtered_values]


# # Add first value from filtered_values_2050 to the beginning of filtered_values_2023

# filtered_values_2023.insert(0, filtered_values_2050[0])


# difference = [v2050 - v2023 for v2050, v2023 in zip(filtered_values_2050, filtered_values_2023)]
# colors = ['green' if d >= 0 else 'orange' for d in difference]
# ax3.bar(filtered_values, difference, alpha=0.7, color=colors)
# ax3.set_title('Difference in Frequencies (2024 age x - 2023 age x-1) ')
# ax3.set_xlabel('Age Values')
# ax3.set_ylabel('Frequency Difference')
# ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
# ax3.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# # Print some statistics about the difference
# print(f"\nDifference Analysis:")
# print(f"Values with positive difference (more in 2050): {sum(1 for d in difference if d > 0)}")
# print(f"Values with negative difference (more in 2023): {sum(1 for d in difference if d < 0)}")
# print(f"Values with no change: {sum(1 for d in difference if d == 0)}")
# print(f"Maximum increase: {max(difference)}")
# print(f"Maximum decrease: {min(difference)}")


In [ ]:
# # step 1 
# matrix_0 = age_matrix_vec_2023[0]
# rows_last_col_not_zero = matrix_0[matrix_0[:, -1] != -1]

# # rows_last_col_not_zero， age_matrix_vec_2050[0][:74999, ]

# # Find range of values in last column of rows_last_col_not_zero
# unique_values = np.unique(rows_last_col_not_zero[:, -1])
# print(f"Range of values in last column: {unique_values.min()} to {unique_values.max()}")
# print(f"Unique values: {unique_values}")

# # Get the second matrix
# second_matrix = age_matrix_vec_2050[0][:second_matrix.shape[0], ]

# # # Perform left join logic
# # result_rows = []
# # for value in unique_values:
# #     # Find rows in first matrix where last column equals value
#     first_matrix_rows = rows_last_col_not_zero[rows_last_col_not_zero[:, -1] == value]
    
#     # Find rows in second matrix where first column equals value + 1
#     second_matrix_rows = second_matrix[second_matrix[:, 0] == (value + 1)]
    
#     # print(f"Value {value}: First matrix has {len(first_matrix_rows)} rows, Second matrix has {len(second_matrix_rows)} rows")
    
#     # # Check if first matrix has fewer rows than second matrix (should not happen)
#     # if len(first_matrix_rows) < len(second_matrix_rows):
#     #     print(f"WARNING: First matrix has more rows ({len(first_matrix_rows)}) than second matrix ({len(second_matrix_rows)}) for value {value}")
    
#     # Efficient left join based on matrix sizes
#     if len(first_matrix_rows) == len(second_matrix_rows):
#         # Block concatenation when sizes match
#         joined_block = np.concatenate([first_matrix_rows, second_matrix_rows], axis=1)
#         result_rows.extend(joined_block)
#     else:
#         # len(first_matrix_rows) > len(second_matrix_rows)
#         # Block join for available matches
#         if len(second_matrix_rows) > 0:
#             matched_first = first_matrix_rows[:len(second_matrix_rows)]
#             joined_block = np.concatenate([matched_first, second_matrix_rows], axis=1)
#             result_rows.extend(joined_block)
        
#         # Handle unmatched rows with -1 padding
#         if len(first_matrix_rows) > len(second_matrix_rows):
#             unmatched_first = first_matrix_rows[len(second_matrix_rows):]
#             padding = np.full((len(unmatched_first), second_matrix.shape[1]), -1)
#             unmatched_block = np.concatenate([unmatched_first, padding], axis=1)
#             result_rows.extend(unmatched_block)

# # Convert to numpy array
# left_joined_matrix = np.array(result_rows)


# print(f"Left joined matrix shape: {left_joined_matrix.shape}")

# # Validation: Check if the 33rd column (last column of first matrix) and 34th column (first column of second matrix) follow the expected pattern
# print("\nValidation of left joined matrix:")
# print(f"Left joined matrix has {left_joined_matrix.shape[1]} columns")
